## LIBRARIES

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## WIDGETS

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:

dbutils.widgets.text("storageName", "saccexplorer")
dbutils.widgets.text("containerName", "bronze")
dbutils.widgets.text("catalogName", "unit_catalog_explorer")
dbutils.widgets.text("schemaName", "uc_bronze")


## CONSTANTS

In [0]:
storage = dbutils.widgets.get("storageName")
container = dbutils.widgets.get("containerName")
catalog =  dbutils.widgets.get("catalogName")
schema =  dbutils.widgets.get("schemaName")

## PATHS

In [0]:
path_base_bronze = f"abfss://{container}@{storage}.dfs.core.windows.net/{schema}"

path_ranking  = f"{path_base_bronze}/supercias_ranking"

## SOURCES

In [0]:

# --- (URLs de Supercias Ecuador) ---
url_ranking = "https://appscvsmovil.supercias.gob.ec/ranking/recursos/bi_ranking.csv"

## STRUCTURES

In [0]:
# import pandas as pd
# df_ranking = pd.read_csv(url_ranking)
# print(df_ranking.dtypes)

ranking_schema = StructType([
    StructField("anio", IntegerType(), True),
    StructField("expediente", IntegerType(), True),
    StructField("posicion_general", DoubleType(), True),
    StructField("cia_imvalores", IntegerType(), True),
    StructField("id_estado_financiero", DoubleType(), True),
    StructField("ingresos_ventas", DoubleType(), True),
    StructField("activos", DoubleType(), True),
    StructField("patrimonio", DoubleType(), True),
    StructField("utilidad_an_imp", DoubleType(), True),
    StructField("impuesto_renta", DoubleType(), True),
    StructField("n_empleados", DoubleType(), True),
    StructField("ingresos_totales", DoubleType(), True),
    StructField("utilidad_ejercicio", DoubleType(), True),
    StructField("utilidad_neta", DoubleType(), True),
    StructField("cod_segmento", DoubleType(), True),
    StructField("ciiu_n1", StringType(), True),
    StructField("ciiu_n6", StringType(), True),
    StructField("liquidez_corriente", DoubleType(), True),
    StructField("prueba_acida", DoubleType(), True),
    StructField("end_activo", DoubleType(), True),
    StructField("end_patrimonial", DoubleType(), True),
    StructField("end_activo_fijo", DoubleType(), True),
    StructField("end_corto_plazo", DoubleType(), True),
    StructField("end_largo_plazo", DoubleType(), True),
    StructField("cobertura_interes", DoubleType(), True),
    StructField("apalancamiento", DoubleType(), True),
    StructField("apalancamiento_financiero", DoubleType(), True),
    StructField("end_patrimonial_ct", DoubleType(), True),
    StructField("end_patrimonial_nct", DoubleType(), True),
    StructField("apalancamiento_c_l_plazo", DoubleType(), True),
    StructField("rot_cartera", DoubleType(), True),
    StructField("rot_activo_fijo", DoubleType(), True),
    StructField("rot_ventas", DoubleType(), True),
    StructField("per_med_cobranza", DoubleType(), True),
    StructField("per_med_pago", DoubleType(), True),
    StructField("impac_gasto_a_v", DoubleType(), True),
    StructField("impac_carga_finan", DoubleType(), True),
    # Indicadores de Rentabilidad
    StructField("rent_neta_activo", DoubleType(), True),
    StructField("margen_bruto", DoubleType(), True),
    StructField("margen_operacional", DoubleType(), True),
    StructField("rent_neta_ventas", DoubleType(), True),
    StructField("rent_ope_patrimonio", DoubleType(), True),
    StructField("rent_ope_activo", DoubleType(), True),
    StructField("roe", DoubleType(), True),
    StructField("roa", DoubleType(), True),
    StructField("fortaleza_patrimonial", DoubleType(), True),
    # Gastos y Costos
    StructField("gastos_financieros", DoubleType(), True),
    StructField("gastos_admin_ventas", DoubleType(), True),
    StructField("depreciaciones", DoubleType(), True),
    StructField("amortizaciones", DoubleType(), True),
    StructField("costos_ventas_prod", DoubleType(), True),
    StructField("deuda_total", DoubleType(), True),
    StructField("deuda_total_c_plazo", DoubleType(), True),
    StructField("total_gastos", DoubleType(), True)
])


## READ SOURCE

In [0]:
# --- Descarga del archivo (Lectura directa desde el driver)
import requests

temp_local_path = "/tmp/ranking_temp.csv"
response = requests.get(url_ranking, verify=False) 

if response.status_code == 200:
    with open(temp_local_path, "wb") as f:
        f.write(response.content)
    print("Archivo descargado localmente en el Driver.")
else:
    raise Exception(f"Error al descargar: {response.status_code}")


# %sh ls -lh /tmp/

## SAVE SOURCE

In [0]:

# --- Lectura con Spark
df_ranking = (spark.read
              .option("header", "True")
              .option("delimiter", ",") 
              .schema(ranking_schema)
              .csv(f"file:{temp_local_path}")) 

# --- Escritura a Delta (Unity Catalog)
df_ranking.write.format("delta") \
    .mode("overwrite") \
    .option("path", path_ranking) \
    .saveAsTable(f"{catalog}.{schema}.supercias_ranking")

print(f"¡Éxito! Tabla creada en {catalog}.{schema}.supercias_ranking")


# --- BLOQUE DE LIMPIEZA ---
import os
if os.path.exists(temp_local_path):
    os.remove(temp_local_path)
    print(f"Archivo temporal {temp_local_path} eliminado del Driver.")


# %sh ls -lh /tmp/